# Recursion: Build the Full Tree Practice Lab

**Learning objective:** Build a tiny decision tree recursively: at each node, either stop (base case) or split into left/right children and continue.

Base cases in this lab:
- node is **pure** (one class)
- node has fewer than `min_samples_split` samples
- `depth` reaches `max_depth`


In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame([
    {"age": 40, "bp": 110, "y": 0},
    {"age": 42, "bp": 115, "y": 0},
    {"age": 45, "bp": 120, "y": 0},
    {"age": 50, "bp": 135, "y": 1},
    {"age": 55, "bp": 140, "y": 1},
    {"age": 60, "bp": 150, "y": 1},
    {"age": 48, "bp": 125, "y": 0},
    {"age": 52, "bp": 145, "y": 1},
])
print(df)


In [ ]:
def gini_impurity(y):
    y = np.asarray(y)
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y)
    props = counts / counts.sum()
    return float(1.0 - np.sum(props ** 2))

def weighted_gini(y_l, y_r):
    n = len(y_l) + len(y_r)
    return 0.0 if n == 0 else (len(y_l) / n) * gini_impurity(y_l) + (len(y_r) / n) * gini_impurity(y_r)

def best_split(frame, features):
    y = frame["y"].to_numpy()
    best = None
    best_score = float("inf")
    for feature in features:
        values = sorted(frame[feature].unique())
        thresholds = [(values[i] + values[i + 1]) / 2 for i in range(len(values) - 1)]
        for t in thresholds:
            left = frame[frame[feature] <= t]
            right = frame[frame[feature] > t]
            if len(left) == 0 or len(right) == 0:
                continue
            score = weighted_gini(left["y"].to_numpy(), right["y"].to_numpy())
            if score < best_score:
                best_score = score
                best = {"feature": feature, "threshold": t, "weighted_gini": score}
    return best

print("best_split helper ready.")


## Guided example — recursive builder

`build_tree` returns either a **leaf** `{type: "leaf", prediction}` or a **node** with left/right children.


In [ ]:
def build_tree(frame, features, depth=0, max_depth=2, min_samples_split=2):
    y = frame["y"].to_numpy()
    # Base cases
    if len(np.unique(y)) == 1:
        return {"type": "leaf", "prediction": int(y[0]), "reason": "pure", "n": len(frame)}
    if len(frame) < min_samples_split:
        pred = int(np.bincount(y).argmax())
        return {"type": "leaf", "prediction": pred, "reason": "min_samples_split", "n": len(frame)}
    if depth >= max_depth:
        pred = int(np.bincount(y).argmax())
        return {"type": "leaf", "prediction": pred, "reason": "max_depth", "n": len(frame)}

    split = best_split(frame, features)
    if split is None:
        pred = int(np.bincount(y).argmax())
        return {"type": "leaf", "prediction": pred, "reason": "no_split", "n": len(frame)}

    left = frame[frame[split["feature"]] <= split["threshold"]]
    right = frame[frame[split["feature"]] > split["threshold"]]
    return {
        "type": "node",
        "feature": split["feature"],
        "threshold": split["threshold"],
        "weighted_gini": round(split["weighted_gini"], 4),
        "left": build_tree(left, features, depth + 1, max_depth, min_samples_split),
        "right": build_tree(right, features, depth + 1, max_depth, min_samples_split),
    }

tree = build_tree(df, features=["age", "bp"], max_depth=2)
print(tree)


## Exercise 1 — inspect children

Write a function `describe(node, indent=0)` that prints each node/leaf. Call it on `tree`.

**Expected:** indented text showing the root split and leaf predictions.


In [ ]:
# TODO Exercise 1
def describe(node, indent=0):
    pad = "  " * indent
    if node["type"] == "leaf":
        print(f"{pad}Leaf n={node['n']} predict={node['prediction']} ({node['reason']})")
        return
    print(f"{pad}Node {node['feature']} <= {node['threshold']} (wg={node['weighted_gini']})")
    print(f"{pad}Left:")
    describe(node["left"], indent + 1)
    print(f"{pad}Right:")
    describe(node["right"], indent + 1)

describe(tree)


## Exercise 2 — change a stopping rule

Rebuild the tree with `max_depth=1` and compare the printed structure to `max_depth=2`.

**Expected:** a shallower tree (root + leaves only).


In [ ]:
# TODO Exercise 2
shallow = build_tree(df, features=["age", "bp"], max_depth=1)
print("max_depth=1 tree:")
describe(shallow)


## Reflection

1. Which base case fired for each leaf in the `max_depth=2` tree?
2. What would happen with `max_depth=10` on this tiny dataset?
3. Why must both left and right children be created after a successful split?
